# Statistics, latency and figure data

Bootstrap confidence intervals, end-to-end latency (Table 13), FakeAVCeleb intervals with hardware details, and the data behind Figures 2, 3 and 6 to 9.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import numpy as np, pandas as pd, json, time, joblib, torch, torch.nn as nn
from pathlib import Path
from PIL import Image
from torchvision import transforms
from torchvision.models import efficientnet_v2_s
from sklearn.metrics import roc_auc_score, accuracy_score
from scipy import stats as sstats

OUT=Path('/content/drive/MyDrive/deepfake_finetuning/rebuild_2026')
RES,MOD,FEA=OUT/'results',OUT/'models',OUT/'features'

def bal(y,s,thr=0):
    p=(s>thr).astype(int)
    tn=((y==0)&(p==0)).sum(); fp=((y==0)&(p==1)).sum()
    fn=((y==1)&(p==0)).sum(); tp=((y==1)&(p==1)).sum()
    return float((tp/(tp+fn)+tn/(tn+fp))/2)

def cluster_ci(y,s,g,fn,n=2000,seed=42):
    u=np.unique(g); ix={k:np.where(g==k)[0] for k in u}
    rb=np.random.default_rng(seed); out=[]
    for _ in range(n):
        rows=np.concatenate([ix[k] for k in rb.choice(u,size=len(u),replace=True)])
        if len(np.unique(y[rows]))<2: continue
        try: out.append(fn(y[rows],s[rows]))
        except Exception: pass
    return float(fn(y,s)),float(np.percentile(out,2.5)),float(np.percentile(out,97.5)),len(out)

ci={}
for name,f,gk in [('ffpp','ffpp_test_scores.npz','videos'),
                  ('dfd','dfd_test_scores.npz','videos'),
                  ('celebdf','celebdf_test_scores.npz','groups')]:
    p=RES/f
    if not p.exists(): print('skip',name); continue
    d=np.load(p,allow_pickle=True); y,s,g=d['y'],d['scores'],d[gk]
    row={}
    for m,fn in [('auc',roc_auc_score),
                 ('accuracy',lambda a,b: accuracy_score(a,(b>0).astype(int))),
                 ('balanced_accuracy',bal)]:
        pt,lo,hi,nb=cluster_ci(y,s,g,fn)
        row[m]=dict(point=pt,ci95=[lo,hi],n_boot=nb)
        print(f'{name:9s} {m:18s} {pt:.4f}  [{lo:.4f}, {hi:.4f}]',flush=True)
    row['clusters']=int(len(np.unique(g))); ci[name]=row

# end-to-end latency, single CPU thread
torch.set_num_threads(1)
m=efficientnet_v2_s(weights=None); m.classifier[1]=nn.Linear(m.classifier[1].in_features,2)
m.load_state_dict(torch.load(MOD/'efficientnetv2s_ffpp_clean.pth',map_location='cpu',
                             weights_only=False)['model_state']); m.eval()
b=joblib.load(MOD/'svm_ffpp_clean.pkl'); scaler,svm=b['scaler'],b['svm']
paths=pd.read_csv(OUT/'frames'/'ffpp_frames.csv').query("partition=='test'").frame_path.head(100).tolist()
TF=transforms.Compose([transforms.Resize((300,300)),transforms.ToTensor(),
   transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
st={k:[] for k in ('load','preprocess','backbone','svm','total')}
for p in paths:
    t0=time.perf_counter(); img=Image.open(p).convert('RGB'); t1=time.perf_counter()
    x=TF(img).unsqueeze(0); t2=time.perf_counter()
    with torch.no_grad(): e=m.avgpool(m.features(x)).flatten(1).numpy()
    t3=time.perf_counter(); svm.decision_function(scaler.transform(e)); t4=time.perf_counter()
    for k,v in zip(('load','preprocess','backbone','svm','total'),
                   [(t1-t0),(t2-t1),(t3-t2),(t4-t3),(t4-t0)]): st[k].append(v*1000)
tm={k:dict(mean_ms=float(np.mean(v)),p95_ms=float(np.percentile(v,95))) for k,v in st.items()}
tm['per_20_frame_video_ms']=tm['total']['mean_ms']*20
tm['note']='single CPU thread; excludes MTCNN face detection'
print('\nlatency per frame (CPU, 1 thread):')
for k in ('load','preprocess','backbone','svm','total'):
    print(f'  {k:11s} {tm[k]["mean_ms"]:8.3f} ms  (p95 {tm[k]["p95_ms"]:.3f})')
print(f'  20-frame video: {tm["per_20_frame_video_ms"]:.0f} ms')
ci['timing']=tm
json.dump(ci,open(RES/'statistics_and_timing.json','w'),indent=2,default=float)
print('\nsaved')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ffpp      auc                0.9848  [0.9787, 0.9898]
ffpp      accuracy           0.9553  [0.9448, 0.9646]
ffpp      balanced_accuracy  0.9373  [0.9209, 0.9518]
dfd       auc                0.9689  [0.9580, 0.9785]
dfd       accuracy           0.9091  [0.8953, 0.9237]
dfd       balanced_accuracy  0.9095  [0.8923, 0.9260]
celebdf   auc                0.6589  [0.6235, 0.6977]
celebdf   accuracy           0.6549  [0.6154, 0.6858]
celebdf   balanced_accuracy  0.5809  [0.5552, 0.6092]



latency per frame (CPU, 1 thread):
  load           2.873 ms  (p95 3.757)
  preprocess     1.262 ms  (p95 1.365)
  backbone     188.921 ms  (p95 192.456)
  svm            2.335 ms  (p95 2.394)
  total        195.390 ms  (p95 199.058)
  20-frame video: 3908 ms

saved


In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_auc_score
RES = Path('/content/drive/MyDrive/deepfake_finetuning/rebuild_2026/results')
d = np.load(RES/'fakeav_test_scores.npz', allow_pickle=True)
rng = np.random.default_rng(42)
for name, y, s in [('visual', d['y_visual'], d['visual']),
                   ('audio',  d['y_audio'],  d['audio']),
                   ('fusion', d['y_visual'], d['fusion'])]:
    a, c = [], []
    for _ in range(2000):
        i = rng.integers(0, len(y), len(y))
        if len(set(y[i])) < 2: continue
        a.append(roc_auc_score(y[i], s[i])); c.append(((s[i] > 0) == y[i]).mean())
    print(f'{name:7s} AUC CI [{np.percentile(a,2.5):.4f}, {np.percentile(a,97.5):.4f}]  '
          f'accuracy CI [{100*np.percentile(c,2.5):.2f}, {100*np.percentile(c,97.5):.2f}]')
print()
!lscpu | grep -E "Model name|^CPU\(s\)"
!free -h | head -2

Mounted at /content/drive
visual  AUC CI [1.0000, 1.0000]  accuracy CI [97.90, 100.00]
audio   AUC CI [0.9688, 1.0000]  accuracy CI [95.10, 100.00]
fusion  AUC CI [1.0000, 1.0000]  accuracy CI [97.90, 100.00]

CPU(s):                                  12
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
               total        used        free      shared  buff/cache   available
Mem:            52Gi       2.3Gi        47Gi       2.4Mi       4.0Gi        50Gi


In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_curve, roc_auc_score
from openpyxl import Workbook

RES = Path('/content/drive/MyDrive/deepfake_finetuning/rebuild_2026/results')
grid = np.linspace(0, 1, 201)

def roc(y, s):
    f, t, _ = roc_curve(y, s)
    return np.interp(grid, f, t)

def cm(y, s, thr):
    p = (s > thr).astype(int)
    return [int(((y==0)&(p==0)).sum()), int(((y==0)&(p==1)).sum()),
            int(((y==1)&(p==0)).sum()), int(((y==1)&(p==1)).sum())]

ff = np.load(RES/'ffpp_test_scores.npz', allow_pickle=True)
cd = np.load(RES/'celebdf_test_scores.npz', allow_pickle=True)
av = np.load(RES/'asvspoof_eval_scores.npz', allow_pickle=True)
fa = np.load(RES/'fakeav_test_scores.npz', allow_pickle=True)

f, t, th = roc_curve(av['y'], av['scores'])
asv_thr = th[int(np.nanargmin(np.abs((1-t) - f)))]      # equal error rate threshold

sets = [('FF++ C23', ff['y'], ff['scores'], 0),
        ('Celeb-DF v2', cd['y'], cd['scores'], 0),
        ('ASVspoof 2019 LA', av['y'], av['scores'], asv_thr),
        ('FakeAVCeleb v1.2 Visual', fa['y_visual'], fa['visual'], 0),
        ('FakeAVCeleb v1.2 Audio', fa['y_audio'], fa['audio'], 0),
        ('FakeAVCeleb v1.2 AV Fusion', fa['y_visual'], fa['fusion'], 0)]

wb = Workbook(); ws = wb.active
for k, (name, y, s, thr) in enumerate(sets):
    c = 1 + 2*k
    ws.cell(1, c, f'{name} FPR'); ws.cell(1, c+1, f'{name} TPR')
    for i, (x, v) in enumerate(zip(grid, roc(y, s)), start=2):
        ws.cell(i, c, float(x)); ws.cell(i, c+1, float(v))
    ws.cell(2+k, 14, name)
    for j, v in enumerate(cm(y, s, thr) + [round(float(roc_auc_score(y, s)), 4)]):
        ws.cell(2+k, 15+j, v)
    print(f'{name:28s} CM {cm(y, s, thr)}  AUC {roc_auc_score(y, s):.4f}')
for j, h in enumerate(['Dataset','TN','FP','FN','TP','AUC']): ws.cell(1, 14+j, h)
ws['U1'], ws['V1'], ws['U2'], ws['V2'], ws['U3'], ws['V3'] = 'Random FPR','Random TPR',0,0,1,1
wb.save(RES/'roc_export.xlsx'); print('saved roc_export.xlsx')

Mounted at /content/drive
FF++ C23                     CM [2548, 251, 499, 13498]  AUC 0.9848
Celeb-DF v2                  CM [1225, 2332, 1242, 5558]  AUC 0.6589
ASVspoof 2019 LA             CM [6139, 1216, 10580, 53302]  AUC 0.9186
FakeAVCeleb v1.2 Visual      CM [86, 0, 1, 56]  AUC 1.0000
FakeAVCeleb v1.2 Audio       CM [102, 2, 1, 38]  AUC 0.9884
FakeAVCeleb v1.2 AV Fusion   CM [86, 0, 1, 56]  AUC 1.0000
saved roc_export.xlsx
